# CSRNet Crowd Density Estimator
### Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU
---
## 🟢 Section A — Quick Demo
**Just want to run the demo? Run cells A1 to A5 only.**

In [ ]:
# A1 — Install dependencies & clone repo
!pip install -q kagglehub tensorboardX gradio
!git clone https://github.com/CommissarMa/CSRNet-pytorch.git
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.3 MB/s eta 0:00:00
Cloning into 'CSRNet-pytorch'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 53 (delta 10), reused 6 (delta 6), pack-reused 34 (from 1)
Receiving objects: 100% (53/53), 19.86 KiB | 6.62 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Done!


In [ ]:
# A2 — Download dataset (used for example images in demo)
import kagglehub
path = kagglehub.dataset_download('tthien/shanghaitech')
print('Dataset path:', path)

100%|██████████| 333M/333M [00:08<00:00, 38.9MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tthien/shanghaitech/versions/1


In [ ]:
# A3 — Download pretrained weights (epoch 73, MAE ~9.4)
import gdown
gdown.download(
    'https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY',
    '/content/73.pth',
    quiet=False
)
print('Weights downloaded!')

Downloading...
From (original): https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY
From (redirected): https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY&confirm=t&uuid=2942d747-f7cd-44b6-b85d-05276ec04596
To: /content/73.pth
100%|██████████| 65.1M/65.1M [00:00<00:00, 176MB/s]

Weights downloaded!


In [ ]:
# A4 — Load model
import sys
sys.path.insert(0, '/content/CSRNet-pytorch')

import torch
from model import CSRNet

model = CSRNet().cuda()
model.load_state_dict(torch.load('/content/73.pth'))
model.eval()
print('Model loaded! MAE ~9.4 on ShanghaiTech Part B')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 96.5MB/s]


Model loaded! MAE ~9.4 on ShanghaiTech Part B


In [ ]:
# A5 — Launch Gradio Demo
import gradio as gr
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
import io, os

def predict(input_img):
    img = Image.fromarray(input_img).convert('RGB')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.225, 0.225, 0.225])
    ])
    img_tensor = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        output = model(img_tensor)
    count = int(output.sum().item())
    density = output.squeeze().cpu().numpy()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(density, cmap='jet')
    ax.axis('off')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    heatmap = Image.open(buf)
    plt.close()
    return heatmap, f'Estimated crowd count: {count} people'

test_img_dir = f'{path}/ShanghaiTech/part_B/test_data/images'
examples = [[os.path.join(test_img_dir, f)] for f in os.listdir(test_img_dir)[:3]]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(label='Upload a crowd image'),
    outputs=[
        gr.Image(label='Density Map'),
        gr.Textbox(label='Count')
    ],
    title='CSRNet Crowd Density Estimator',
    description='Upload a crowd image to estimate the number of people using CSRNet. Best results with overhead/surveillance style images.',
    examples=examples
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9cdfd9042e65227a8a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🟡 Section B — Train from Scratch (Optional)
**Skip this section if you already ran Section A.**

Run this if you want to retrain the model yourself. Google Drive required to save checkpoints.

In [ ]:
# B1 — Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/CSRNet/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive mounted! Checkpoints will save to:', CKPT_DIR)

In [ ]:
# B2 — Generate density maps (~2 mins)
import os
import numpy as np
import scipy.io as io
from scipy.ndimage import gaussian_filter
from PIL import Image

DMAP_DIR = '/content/densitymaps'
os.makedirs(f'{DMAP_DIR}/train', exist_ok=True)
os.makedirs(f'{DMAP_DIR}/test', exist_ok=True)

def generate_density_map(image, points, sigma=15):
    density = np.zeros(image.shape[:2], dtype=np.float32)
    for pt in points:
        x, y = int(pt[0]), int(pt[1])
        if y < image.shape[0] and x < image.shape[1]:
            density[y, x] += 1.
    return gaussian_filter(density, sigma=sigma)

def process_split(part, split):
    img_dir = f'{path}/ShanghaiTech/{part}/{split}_data/images'
    gt_dir  = f'{path}/ShanghaiTech/{part}/{split}_data/ground-truth'
    out_dir = f'{DMAP_DIR}/{split}'
    imgs = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    already = set(f.replace('.npy','') for f in os.listdir(out_dir))
    remaining = [f for f in imgs if f.replace('.jpg','') not in already]
    print(f'{part} {split}: {len(already)} already done, {len(remaining)} remaining')
    for i, img_file in enumerate(remaining):
        idx = img_file.replace('IMG_','').replace('.jpg','')
        gt_file = os.path.join(gt_dir, f'GT_IMG_{idx}.mat')
        if not os.path.exists(gt_file):
            continue
        img = np.array(Image.open(os.path.join(img_dir, img_file)))
        mat = io.loadmat(gt_file)
        points = mat['image_info'][0][0][0][0][0].astype(float)
        density = generate_density_map(img, points)
        np.save(os.path.join(out_dir, f'IMG_{idx}.npy'), density)
        if i % 50 == 0:
            print(f'  Progress: {i}/{len(remaining)}')
    print(f'Done: {part} {split}')

process_split('part_B', 'train')
process_split('part_B', 'test')
print('All density maps ready!')

In [ ]:
# B3 — Patch dataset.py & configure training
with open('/content/CSRNet-pytorch/dataset.py', 'r') as f:
    old = f.read()
new = old.replace(
    "self.dmap_path = os.path.join(root, phase+'_data/densitymaps')",
    "self.dmap_path = '/content/densitymaps/' + phase"
)
with open('/content/CSRNet-pytorch/dataset.py', 'w') as f:
    f.write(new)
print('dataset.py patched!')

config_content = f'''import torch
import os
from tensorboardX import SummaryWriter

class Config():
    def __init__(self):
        self.dataset_root = "{path}/ShanghaiTech/part_B"
        self.device       = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.lr           = 1e-5
        self.batch_size   = 1
        self.epochs       = 75
        self.checkpoints  = "{CKPT_DIR}"
        self.writer       = SummaryWriter()
        self.__mkdir(self.checkpoints)

    def __mkdir(self, path):
        if not os.path.exists(path):
            os.makedirs(path)
            print("create dir: ", path)
'''
with open('/content/CSRNet-pytorch/config.py', 'w') as f:
    f.write(config_content)
print('Config saved! Training for 75 epochs, checkpoints → Google Drive')

In [ ]:
# B4 — Train (~5 hours, keep tab open)
%cd /content/CSRNet-pytorch
!python train.py

---
## 🔵 Section C — Load & Run after Training
**Run this after Section B finishes.**

Check the training output and note the epoch with the lowest MAE.

In [ ]:
# C1 — Load best checkpoint
import sys
sys.path.insert(0, '/content/CSRNet-pytorch')
import torch, os
from model import CSRNet

CKPT_DIR = '/content/drive/MyDrive/CSRNet/checkpoints'
ckpts = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pth')]
best = sorted(ckpts, key=lambda x: int(x.replace('.pth','')))[-1]
print('Available checkpoints:', sorted(ckpts, key=lambda x: int(x.replace('.pth',''))))
print('Loading:', best, '(update manually if needed)')

model = CSRNet().cuda()
model.load_state_dict(torch.load(f'{CKPT_DIR}/{best}'))
model.eval()
print('Model loaded!')

In [ ]:
# C2 — Launch Gradio Demo
import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
import io, os

def predict(input_img):
    img = Image.fromarray(input_img).convert('RGB')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.225, 0.225, 0.225])
    ])
    img_tensor = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        output = model(img_tensor)
    count = int(output.sum().item())
    density = output.squeeze().cpu().numpy()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(density, cmap='jet')
    ax.axis('off')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    heatmap = Image.open(buf)
    plt.close()
    return heatmap, f'Estimated crowd count: {count} people'

test_img_dir = f'{path}/ShanghaiTech/part_B/test_data/images'
examples = [[os.path.join(test_img_dir, f)] for f in os.listdir(test_img_dir)[:3]]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(label='Upload a crowd image'),
    outputs=[
        gr.Image(label='Density Map'),
        gr.Textbox(label='Count')
    ],
    title='CSRNet Crowd Density Estimator',
    description='Upload a crowd image to estimate the number of people using CSRNet. Best results with overhead/surveillance style images.',
    examples=examples
)

demo.launch(share=True)